## **LangChain React Agent With `tools`**

In [1]:
from langchain.tools import tool

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1-mini")

In [4]:
## best practice tools
@tool
def square_root(x: float) -> float:
    """Calculate the square of the number."""

    return x ** 2

@tool("add")
def tool2(a: float, b: float) -> float:
    """Calculate the sum of the two number."""

    return a + b

In [5]:
## tools executions.
square_root.invoke(
    {
        "x": 5
    }
)

25.0

## **Agent initializations with `tools`**

In [6]:
from langchain.agents import create_agent

arithmetic_agent = create_agent(
    model=model,
    tools=[square_root, tool2],
    system_prompt="You're a arithmetic wizard. Use the the square tool to square any number, and add tools for additions any two float number"
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Can add this tow number as 2.0 and 3.0 and finally find the squared to the result.")

response = arithmetic_agent.invoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

The sum of 2.0 and 3.0 is 5.0, and the square of this result is 25.0.


## **Agent scramming**

In [8]:
for token, metadata in arithmetic_agent.stream(
    {
        "messages": [
            question
        ]
    },
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)

5.025.0The sum of 2.0 and 3.0 is 5.0, and the square of 5.0 is 25.0.

## **Add `Search web` tools with the agent**

In [9]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_clint = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for the information."""

    return tavily_clint.search(query)

web_search.invoke("Who is the current govt. in the bangladesh?")

{'query': 'Who is the current govt. in the bangladesh?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Government_of_Bangladesh',
   'title': 'Government of Bangladesh',
   'content': 'After the Resignation of Sheikh Hasina in August 2024, the current Interim government is led by Muhammad Yunus as chief adviser. ... List of political parties in ...Read more',
   'score': 0.99946004,
   'raw_content': None},
  {'url': 'https://globaledge.msu.edu/countries/bangladesh/government',
   'title': 'Bangladesh: Government - globalEDGE',
   'content': "Key Figures. Chief of State: President Mohammad Shahabuddin Chuppi; Head of Government: Currently vacant. Overview. Government Name: People's Republic of ...Read more",
   'score': 0.9975466,
   'raw_content': None},
  {'url': 'https://commonslibrary.parliament.uk/research-briefings/cbp-10096/',
   'title': 'Bangladesh: The fall of the Hasina Government and recent ...',
   'conten

In [ ]:
smart_web_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="You've web search tools so if user ask a questions but need realtime info the search in the web for accurate result."
)

In [18]:
question = "Who is the current govt. in the bangladesh?"
response = smart_web_agent.invoke(
    {"messages":[HumanMessage(content=question)]}
)

response["messages"][-1].content

'The current government in Bangladesh is an interim government led by Muhammad Yunus as the chief adviser. This change occurred after the resignation of Sheikh Hasina in August 2024.'